# This file gathers COUNTER data from vendors who offer usage data by directly accessing their COUNTER APIs.

## This version can only download one report at a time. It is best used if you are just starting out in usage, want full control of when usage is run, and/or you want a small number of files only.

In [22]:
# Load Packages

import numpy as np
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
import shutil
from pprint import pprint
import csv
import sqlite3
from sqlite3 import Error
from pathlib import Path
import ast
import warnings
import xlsxwriter
import requests as req
import re
import json


warnings.filterwarnings("ignore") #This is to improve the code's performance. If you are troubleshooting the code, remove this line first.

## SUSHI API Data by Vendor is saved in the API_Info.py file. If any APIs need updates, make changes within that file.

## Load the package in the cell below, either by connecting to the folder it is stored or putting API_Info.py in the same folder as this code.

# NOTE: Do not post that API_Info file to GitHub at any time! It must be stored in a safe, non-public location. 

In [23]:
#Use this if the API_Info file is not in the same folder as the code - otherwise, the package can just be imported
apiInfoFolderPath = r"C:\Users\aharrima\GitHub_UTK\Usage_Database_Holding_Area"
directory = os.chdir(apiInfoFolderPath)

import API_Info

# Input Key Variables for the Vendor

In [24]:
vendor = 'OUP'  #This should be the Naming Convention, and must match the start of each portion of the dictionary within API_Info.
startDate = '2026-07-01' #Please use YYYY-MM-DD format for both dates; while some vendors only need YYYY-MM, the full date is used later in this code.
endDate = '2026-08-31'
reportType = 'TR_B3' #Include the underscore if you are gathering reports other than the Master Report. Example: IR_M1, PR_P1

## Gather Data

In [25]:
# COUNTER 5 or 5.1 Check
if startDate >= '2025-01-01':
    counterType = 5.1
else:
    counterType = 5


if vendor == 'ACS' or vendor == 'AmerPhysicalSoc' or vendor == 'APA' or vendor == 'Cabells_Journalytics': #Check for vendors that only offer COUNTER 5.1
    counterPiece = 'r51/reports/'
elif vendor == 'IngentaConnect': #Check for vendors that do not include the "reports" call
    counterPiece = ''
elif counterType == 5.1 and vendor != 'AdamMatthew' and vendor != 'ASP' and vendor != 'BiblioScout' and vendor != 'Cambridge' and vendor != 'CSIRO' and vendor != 'D&H' and vendor != 'Highwire' and vendor != 'IOP' and vendor != 'ISIWebofScience' and vendor != 'MohrSiebeck' and vendor != 'NewsBank' and vendor != 'RSC' and vendor != 'SAE': #COUNTER 5.1 check, along with vendors that only offer COUNTER 5
    counterPiece = 'r51/reports/'
else:
    counterPiece = 'reports/'

# Requestor Check
if API_Info.requestorIDs[vendor] == '':
    requestorPiece = ''
else:
    requestorPiece = r''+'requestor_id='+API_Info.requestorIDs[vendor]

# Customer Check
if API_Info.customerIDs[vendor] == '':
    customerPiece = ''
else:
    customerPiece = r''+'&customer_id='+API_Info.customerIDs[vendor]

# Customer First Parameter Check
if vendor == 'Thieme' or vendor == 'SAE' or vendor == 'IEEE':
    customerPiece = r''+'customer_id='+API_Info.customerIDs[vendor]

# API Key Check
if API_Info.apiKeys[vendor] == '':
    apiKeyPiece = ''
else:
    apiKeyPiece = r''+'&api_key='+API_Info.apiKeys[vendor]

# Slash Piece Correction
if vendor == 'Gale61277' or vendor == 'ProjectMuse' or vendor == 'Thieme' or vendor == 'Wiley' or vendor == 'APA' or vendor == 'AMS' or vendor == 'AmerPhysicalSoc' or vendor == 'BiblioScout' or vendor == 'BOA' or vendor == 'Brepols' or vendor == 'BrillOnline' or vendor == 'Cabells_Journalytics' or vendor == 'CHE_JR' or vendor == 'CIAO' or vendor == 'Credo' or vendor == 'DeGruyter' or vendor == 'EastView' or vendor == 'FirstSearch' or vendor == 'Highwire' or vendor == 'HistoryCommons' or vendor == 'IngentaConnect' or vendor == 'JBE' or vendor == 'JSTAGE' or vendor == 'LOEB' or vendor == 'MicrobiologySociety' or vendor == 'MohrSiebeck' or vendor == 'MSP' or vendor == 'PLOS' or vendor == 'ProjectMuse' or vendor == 'ProQuestEBooks' or vendor == 'RDAToolkit' or vendor == 'Rotunda' or vendor == 'SAE':
    slashPiece = '?'
else:
    slashPiece = '/?'

# Mandatory Lowercase Vendor Check
if vendor == 'AccessEngineering' or vendor == 'ASTMCompass' or vendor == 'BrillOnline' or vendor == 'SpringerAll' or vendor == 'Thieme' or vendor == 'Wiley' or vendor == 'ISIWebofScience' or vendor == 'EBSCO' or vendor == 'Credo' or vendor == 'BiblioScout' or vendor == 'Brepols' or vendor == 'JBE' or vendor == 'JSTAGE' or vendor == 'MicrobiologySociety' or vendor == 'MohrSiebeck' or vendor == 'RSC' or vendor == 'SAE' or vendor == 'Cabells_Journalytics' or vendor == 'IEEE' or vendor == 'IOP' or vendor == 'IngentaConnect':
    reportType = reportType.lower()


# Mandatory YYYY-MM Vendor Check
if vendor == 'JSTAGE' or vendor == 'IngentaConnect':
    startDate = startDate[:7]
    endDate = endDate[:7]

# Mandatory Platform Check


requestURL = r''+API_Info.urls[vendor]+counterPiece+reportType+slashPiece+requestorPiece+customerPiece+apiKeyPiece+'&begin_date='+startDate+'&end_date='+endDate

rawRequest = req.get(requestURL)

rawRequest

<Response [200]>

In [26]:
requestURL

'https://sushi5.scholarlyiq.com/counter/r51/reports/TR_B3/?requestor_id=93c2b65b-7235-416f-9c8b-1255dce720ca&customer_id=20063667&begin_date=2026-07-01&end_date=2026-08-31'

#### If you see "Response [200]" above, then the report loaded properly and you can continue.

#### If any other number appears in the brackets, then an error occurred and no data was downloaded.

#### Common errors and what they mean:

1. **400, 401, 404** The requestURL and/or SUSHI information listed in the dictionary is incorrect. Fix the URL or SUSHI information as needed, then try again.
3. **403** The vendor uses a third-party platform for usage that is not conducive to gathering via Python. Silverchair is the most common culprit. You have options for your next steps (see below).
4. **202** The server is working on gathering data but needs some time to download it. If you see a 202 error, wait and try again; after enough time passes (often a few seconds but can be longer), you should be good to go. Note that longer reports often require longer wait times. LibLynx is the most common culprit for 202 messages.
6. **500** The server can not fulfill the request, either because there are too many requests in a short period of time, the server is not active, or there are not enough pieces of information to reach the API with your URL. Your response to these should be as follows:
    1. **Too Many Requests** - Wait for some time and try again. If it continues to fail, gather by hand.
    2. **Non-Active Server** - You can check to see how long the API may be offline if you want. Then, either wait and try again or gather by hand.
    3. **Not Enough Info** - Mostly comes from troubleshooting APIs. Add more parameters and try again.

### Did you get a "Response [403]" from the above cell? There may still be a way to gather SUSHI in this code!

Some vendors do not allow Python as a method of gathering usage, since those vendors see Python as a breeding ground for AI and other forms of indiscriminant data scraping. However, we are still able to get SUSHI data from these vendors; it will just take a few more manual steps.

If you see a 403 error, you can either gather usage via the vendor's website, or see if it exists by going to the requestURL listed above (it should be an active URL, so you can just click on it). Your next steps depend on what happens:


1. If a webpage appeared where the usage data is shown in a bunch of brackets, then copy this information EXACTLY AS IT APPEARS into Notepad. Save as a .json file  (you will need to Save As "All Files" and add .json to the end of your file name to do this). Then, load it using the manual JSON file cell below.
2. If a data file was automatically downloaded, open it up. Copy the data in cell A1 EXACTLY AS IT APPEARS into Notepad. Save as a .json file (you will need to Save As "All Files" and add .json to the end of your file name to do this). Then, load it using the manual JSON file cell below.
   
**NOTE: Excel can only store 32,760 characters in a cell. This means for reports spanning long periods of time and/or reports with lots of information (such as a large TR) this approach may not be viable.**
**If you copy data from Excel into Notepad and see it is 32,760 characters long, either shorten the time period being gathered or go to the website to gather by hand instead.**

3. If the usage data does not appear when clicking on the URL, then go to the vendor's website and manually gather the data.

**Note** that this method may take longer than simply gathering by hand, depending on how good the vendor's usage platform is.

## Convert the Data from JSON into a Data Table

JSON is the standard format for downloading data from an API, but it is not user-friendly. We will convert the JSON into an Excel file by extracting the important information.

### If you saved the data into a JSON file manually (usually because you got a Response [403] message), use the cell below to load it.

In [27]:
#with open('WileyJSONJul2026Aug2026.json', 'r', encoding='utf-8') as file:
#    data = json.load(file)

### If you were able to successfully gather the data directly in Python (you got a Response [200] message), use the cell below to load it instead.

In [28]:
data = rawRequest.json()

### Regardless of loading method, run all cells below to begin usage cleanup.

In [29]:
# Remove any underscores from report names for naming conventions later
if '_' in reportType:
    reportType = re.sub(r'[^a-zA-Z0-9]', '', reportType)

In [30]:
data['Report_Header']

{'Created': '2026-09-21T12:05:41Z',
 'Created_By': 'Scholarly iQ for Oxford University Press',
 'Report_ID': 'TR_B3',
 'Release': '5.1',
 'Report_Name': 'Book Usage by Access Type',
 'Registry_Record': 'https://registry.countermetrics.org/platform/1860d58d-dc85-4105-8477-910a56025cc1',
 'Institution_Name': 'University of Tennessee Libraries',
 'Institution_ID': {'Proprietary': ['OUP:20063667']},
 'Report_Filters': {'Access_Method': ['Regular'],
  'Data_Type': ['Book', 'Reference_Work'],
  'Metric_Type': ['Total_Item_Investigations',
   'Total_Item_Requests',
   'Unique_Item_Investigations',
   'Unique_Item_Requests',
   'Unique_Title_Investigations',
   'Unique_Title_Requests'],
  'Begin_Date': '2026-07-01',
  'End_Date': '2026-08-31'}}

In [31]:
if 'Exceptions' in data['Report_Header']:
    print(f"Exceptions Found: {data['Report_Header']['Exceptions']}")
    raise ValueError('An Exception was found gathering this report.')

### Check for Exceptions Before Continuing!

SUSHI has a few exceptions built into it that will not prevent Python from gathering data like the messages above, but may lead to inaccurate or incomplete data. They will appear in the Report Header, shown above.

#### Some common Exceptions include:

1. **3030 OR 3032** - Usage is not available for certain dates. This often means one of two cases:
    1. The report is not offered by the vendor. If so, skip and move to the next report.
    2. The usage is too old to appear in the report. Some vendors only hold a certain amount of older usage data, with only a few holding more than 3 years. If this is the case, either shorten your gathering window or continue with the available usage in your window.
3. **3000** - The report you asked for is not supported (most likely, a Database Report or the IR_M1 causes this). In this case, the "dataItems" cell below should have no information, and you can move on to the next vendor/report.

Note that sometimes these exceptions are the only info that appear in the data variable and are not in the Report_Header section.

In [32]:
dataItems = data['Report_Items']

dataItems[0]

{'Title': 'A Dictionary of',
 'Publisher': 'Oxford University Press',
 'Publisher_ID': {'Proprietary': ['OUP:OUP']},
 'Item_ID': {'ISBN': '978-0-19-958505-2'},
 'Platform': 'Oxford Reference',
 'Attribute_Performance': [{'Access_Type': 'Controlled',
   'YOP': '2010',
   'Performance': {'Total_Item_Investigations': {'2026-07': 1},
    'Total_Item_Requests': {'2026-07': 1},
    'Unique_Item_Investigations': {'2026-07': 1},
    'Unique_Item_Requests': {'2026-07': 1},
    'Unique_Title_Investigations': {'2026-07': 1},
    'Unique_Title_Requests': {'2026-07': 1}},
   'Data_Type': 'Book'}]}

#### If the above cell did not return information about a platform, database, title, or item, then there may be an error from the data-side instead of the server-side. Run the cell below to see if there were any issues.

Common reasons why the JSON did not download properly:

1. You are requesting data from a date range the vendor does not have stored. This should appear as an Exception in the 3000s range when the above cell is run, and it should also explain what data is available.
2. The data is not available in the COUNTER type requested. This results in the same exception as #1, requiring either a change in date range or a change in data type.
3. You are asking for a report type the vendor does not offer. The Usage Master Vendor List should contain information about each report type, as will the vendor's website.
4. Sometimes, inaccurate SUSHI information will still hit the server but return no data, leading to an Exception here instead of in the first request box.
5. The report has formatting issues. These are rare but should be reported to the vendor. All COUNTER reports should have consistent formatting. It will be your choice to either wait, edit the report and continue, or gather by hand.

# Convert JSON Files Into a Pandas DataFrame


### This version converts the JSON into an Excel-style table, or a DataFrame in Python lingo. This is what will be sent to Excel after all formatting is completed.

##### Clear out all previous data information and create the columns

In [41]:
Platform = pd.Series([])
DataType = pd.Series([])
MetricType = pd.Series([])
Month = pd.Series([])
Usage = pd.Series([])
Vendor = pd.Series([])
Database = pd.Series([])
Title = pd.Series([])
OnlineISSN = pd.Series([])
PrintISSN = pd.Series([])
ISBN = pd.Series()
Publisher = pd.Series([])
AccessType = pd.Series([])
Item = pd.Series([])
onlineIssnName = 'NA'
printIssnName = 'NA'
isbnName = 'NA'

## Checking for Various Types of JSON Downloads (this is purely a backend thing)


if 'Attribute_Performance' in dataItems or 'Attribute_Performance' in dataItems[0]:# or 'Attribute_Performance' in dataItems[0]['Items'][0]: #This third one is specifically for IR_M1 formatting, and needs to be commented out if a different report is running.
    functionVersion = 'Attribute_Performance'
else:
    functionVersion = 'Performance'

if 'Data_Type' in dataItems[0]:
    dataTypeVersion = 'Data_Type'
else:
    dataTypeVersion = 'Unknown'

In [42]:
functionVersion

'Attribute_Performance'

#### Create the DataFrame

In [43]:
# Platform Report Downloads - PR or PR_P1

if reportType == 'PR' or reportType == 'pr' or reportType == 'PRP1' or reportType == 'prp1':
    if functionVersion == 'Attribute_Performance':
        for h in range(len(dataItems)):
            attributes = dataItems[h]['Attribute_Performance']
            platformName = dataItems[h]['Platform']
            for i in range(len(attributes)):
                performanceList = attributes[i]['Performance']
                dataType = attributes[i]['Data_Type']
                for j in range(len(performanceList)):
                    dictionaryItemsJ = performanceList.items()
                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                    metricType = keys[j]
                    months = performanceList[keys[j]]
                    for k in range(len(months)):
                        dictionaryItemsK = months.items()
                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                        month = keys2[k]
                        values = months[keys2[k]]
                        
                        #print('Platform:', platformName)
                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                        #print('Data Type:', dataType)
                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                        #print('Metric Type:', metricType)
                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                        #print('Month:', month)
                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                        #print('Usage Value:', values)
                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
        
                        Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)
    
    if functionVersion == 'Performance':
        for h in range(len(dataItems)):
            performances = dataItems[h]['Performance']
            platformName = dataItems[h]['Platform']
            for i in range(len(performances)):
                performanceList = performances
                month = performances[i]['Period']['Begin_Date']
                dataType = performances[i]['Instance']
                if dataTypeVersion == 'Data_Type':
                    dataName = dataItems[h]['Data_Type']
                else:
                    dataName = 'Unknown'
                for j in range(len(dataType)):
                    metricType = dataType[j]['Metric_Type']
                    values = dataType[j]['Count']
                    
                    #print('Platform:', platformName)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Data Type:', dataType)
                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)    
    
    
    longUsage = pd.concat([Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
    
    longUsage.columns = ['Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']



# Database Report Downloads - DR or DR_D1
if reportType == 'DR' or reportType == 'DRD1' or reportType == 'dr' or reportType == 'drd1':
    if functionVersion == 'Attribute_Performance':
        for h in range(len(dataItems)):
            attributes = dataItems[h]['Attribute_Performance']
            databaseName = dataItems[h]['Database']
            platformName = dataItems[h]['Platform']
            for i in range(len(attributes)):
                performanceList = attributes[i]['Performance']
                if dataTypeVersion == 'Data_Type':
                    dataType = attributes[i]['Data_Type']
                else:
                    dataType = 'Unknown'
                for j in range(len(performanceList)):
                    dictionaryItemsJ = performanceList.items()
                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                    metricType = keys[j]
                    months = performanceList[keys[j]]
                    for k in range(len(months)):
                        dictionaryItemsK = months.items()
                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                        month = keys2[k]
                        values = months[keys2[k]]
                        
                        #print('Platform:', platformName)
                        Database = pd.concat([Database, pd.Series([databaseName])], ignore_index = True)
                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                        #print('Data Type:', dataType)
                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                        #print('Metric Type:', metricType)
                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                        #print('Month:', month)
                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                        #print('Usage Value:', values)
                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
        
                        Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)
    
    
    if functionVersion == 'Performance':
        for h in range(len(dataItems)):
            performances = dataItems[h]['Performance']
            databaseName = dataItems[h]['Database']
            platformName = dataItems[h]['Platform']
            for i in range(len(performances)):
                performanceList = performances
                month = performances[i]['Period']['Begin_Date']
                dataType = performances[i]['Instance']
                if dataTypeVersion == 'Data_Type':
                    dataName = dataItems[h]['Data_Type']
                else:
                    dataName = 'Unknown'
                for j in range(len(dataType)):
                    metricType = dataType[j]['Metric_Type']
                    values = dataType[j]['Count']
                    
                    #print('Platform:', platformName)
                    Database = pd.concat([Database, pd.Series([databaseName])], ignore_index = True)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Data Type:', dataType)
                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)    
    
    
    longUsage = pd.concat([Database, Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
    
    longUsage.columns = ['Database', 'Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']


# Title Master Report Downloads - TR
if reportType == 'TR' or reportType == 'tr':  
    if functionVersion == 'Attribute_Performance':
        for h in range(len(dataItems)):
            if vendor == 'AAAS' and h == 0:
                continue
            onlineIssnName = 'NA'
            printIssnName = 'NA'
            isbnName = 'NA'
            attributes = dataItems[h]['Attribute_Performance']
            titleName = dataItems[h]['Title']
            platformName = dataItems[h]['Platform']
            if 'Publisher' in dataItems[h]:
                publisherName = dataItems[h]['Publisher']
            else:
                publisherName = 'NA'
            if 'Item_ID' in dataItems[h]:
                itemIDs = dataItems[h]['Item_ID']
                if 'Online_ISSN' in itemIDs:
                    onlineIssnName = itemIDs['Online_ISSN']
                if 'Print_ISSN' in itemIDs:
                    printIssnName = itemIDs['Print_ISSN']
                if 'ISBN' in itemIDs:
                    isbnName = itemIDs['ISBN']
            else:
                onlineIssnName = 'NA'
                printIssnName = 'NA'
                isbnName = 'NA'
            for i in range(len(attributes)):
                performanceList = attributes[i]['Performance']
                dataType = attributes[i]['Data_Type']
                for j in range(len(performanceList)):
                    dictionaryItemsJ = performanceList.items()
                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                    metricType = keys[j]
                    months = performanceList[keys[j]]
                    for k in range(len(months)):
                        dictionaryItemsK = months.items()
                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                        month = keys2[k]
                        values = months[keys2[k]]
                        
                        #print('Platform:', platformName)
                        Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                        OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                        PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                        ISBN = pd.concat([ISBN, pd.Series([isbnName])], ignore_index = True)
                        Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                        #print('Data Type:', dataType)
                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                        #print('Metric Type:', metricType)
                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                        #print('Month:', month)
                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                        #print('Usage Value:', values)
                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
        
                        Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)
    
    
    if functionVersion == 'Performance':
        for h in range(len(dataItems)):
            onlineIssnName = 'NA'
            printIssnName = 'NA'
            isbnName = 'NA'
            performances = dataItems[h]['Performance']
            titleName = dataItems[h]['Title']
            if 'Publisher' in dataItems[h]:
                publisherName = dataItems[h]['Publisher']
            else:
                publisherName = 'NA'
            if 'Item_ID' in dataItems[h]:
                itemIDs = dataItems[h]['Item_ID']
                for g in range(len(itemIDs)):
                    if itemIDs[g]['Type'] == 'Online_ISSN':
                        onlineIssnName = itemIDs[g]['Value']
                    if itemIDs[g]['Type'] == 'Print_ISSN':
                        printIssnName = itemIDs[g]['Value']
                    if itemIDs[g]['Type'] == 'ISBN':
                        isbnName = itemIDs[g]['Value']
            else:
                onlineIssnName = 'NA'
                printIssnName = 'NA'
                isbnName = 'NA'
            if 'Platform' in dataItems[h]:
                platformName = dataItems[h]['Platform']
            else:
                platformName = 'NA'
            for i in range(len(performances)):
                performanceList = performances
                month = performances[i]['Period']['Begin_Date']
                dataType = performances[i]['Instance']
                if dataTypeVersion == 'Data_Type':
                    dataName = dataItems[h]['Data_Type']
                else:
                    dataName = 'Unknown'
                for j in range(len(dataType)):
                    metricType = dataType[j]['Metric_Type']
                    values = dataType[j]['Count']
                    
                    #print('Platform:', platformName)
                    Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                    OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                    PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                    ISBN = pd.concat([ISBN, pd.Series([isbnName])], ignore_index = True)
                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Data Type:', dataType)
                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)    
    
    
    longUsage = pd.concat([Title, OnlineISSN, PrintISSN, ISBN, Publisher, Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
    
    longUsage.columns = ['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Publisher', 'Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']

    
# Title Master Report Downloads - TR_B3 and TR_J3
if reportType == 'TRB3' or reportType == 'TRJ3' or reportType == 'trb3' or reportType == 'trj3':        
    if functionVersion == 'Attribute_Performance':
        if vendor == 'AAAS':
            for h in range(1, len(dataItems)):
                attributes = dataItems[h]['Attribute_Performance']
                titleName = dataItems[h]['Title']
                if 'Publisher' in dataItems[h]:
                    publisherName = dataItems[h]['Publisher']
                else:
                    publisherName = 'NA'
                platformName = dataItems[h]['Platform']
                if 'Item_ID' in dataItems[h]:
                    itemIDs = dataItems[h]['Item_ID']
                    if 'Online_ISSN' in itemIDs:
                        onlineIssnName = itemIDs['Online_ISSN']
                    if 'Print_ISSN' in itemIDs:
                        printIssnName = itemIDs['Print_ISSN']
                    if 'ISBN' in itemIDs:
                        isbnName = itemIDs['ISBN']
                else:
                    onlineIssnName = 'NA'
                    printIssnName = 'NA'
                    isbnName = 'NA'
                for i in range(len(attributes)):
                    performanceList = attributes[i]['Performance']
                    accessType = attributes[i]['Access_Type']
                    #dataType = attributes[i]['Data_Type']
                    for j in range(len(performanceList)):
                        dictionaryItemsJ = performanceList.items()
                        keys = list(map(lambda x: x[0], dictionaryItemsJ))
                        metricType = keys[j]
                        months = performanceList[keys[j]]
                        for k in range(len(months)):
                            dictionaryItemsK = months.items()
                            keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                            month = keys2[k]
                            values = months[keys2[k]]
                            
                            #print('Platform:', platformName)
                            Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                            Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                            Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                            #print('Data Type:', dataType)
                            #DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                            AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                            #print('Metric Type:', metricType)
                            MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                            #print('Month:', month)
                            Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                            #print('Usage Value:', values)
                            Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
            
                            Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)
        else:
            for h in range(len(dataItems)):
                attributes = dataItems[h]['Attribute_Performance']
                titleName = dataItems[h]['Title']
                if 'Publisher' in dataItems[h]:
                    publisherName = dataItems[h]['Publisher']
                else:
                    publisherName = 'NA'
                platformName = dataItems[h]['Platform']
                if 'Item_ID' in dataItems[h]:
                    itemIDs = dataItems[h]['Item_ID']
                    if 'Online_ISSN' in itemIDs:
                        onlineIssnName = itemIDs['Online_ISSN']
                    if 'Print_ISSN' in itemIDs:
                        printIssnName = itemIDs['Print_ISSN']
                    if 'ISBN' in itemIDs:
                        isbnName = itemIDs['ISBN']
                else:
                    onlineIssnName = 'NA'
                    printIssnName = 'NA'
                    isbnName = 'NA'
                for i in range(len(attributes)):
                    performanceList = attributes[i]['Performance']
                    accessType = attributes[i]['Access_Type']
                    #dataType = attributes[i]['Data_Type']
                    for j in range(len(performanceList)):
                        dictionaryItemsJ = performanceList.items()
                        keys = list(map(lambda x: x[0], dictionaryItemsJ))
                        metricType = keys[j]
                        months = performanceList[keys[j]]
                        for k in range(len(months)):
                            dictionaryItemsK = months.items()
                            keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                            month = keys2[k]
                            values = months[keys2[k]]
                            
                            #print('Platform:', platformName)
                            Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                            Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                            Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                            #print('Data Type:', dataType)
                            #DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                            AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                            #print('Metric Type:', metricType)
                            MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                            #print('Month:', month)
                            Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                            #print('Usage Value:', values)
                            Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
            
                            Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)            
    
    
    if functionVersion == 'Performance':
        onlineIssnName = 'NA'
        printIssnName = 'NA'
        for h in range(len(dataItems)):
            performances = dataItems[h]['Performance']
            titleName = dataItems[h]['Title']
            publisherName = dataItems[h]['Publisher']
            if 'Item_ID' in dataItems[h]:
                itemIDs = dataItems[h]['Item_ID']
                for g in range(len(itemIDs)):
                    if itemIDs[g]['Type'] == 'Online_ISSN':
                        onlineIssnName = itemIDs[g]['Value']
                    if itemIDs[g]['Type'] == 'Print_ISSN':
                        printIssnName = itemIDs[g]['Value']
            else:
                onlineIssnName = 'NA'
                printIssnName = 'NA'
            accessType = dataItems[h]['Access_Type']
            if 'Platform' in dataItems[h]:
                platformName = dataItems[h]['Platform']
            else:
                platformName = 'NA'
            for i in range(len(performances)):
                performanceList = performances
                month = performances[i]['Period']['Begin_Date']
                dataType = performances[i]['Instance']
                if dataTypeVersion == 'Data_Type':
                    dataName = dataItems[h]['Data_Type']
                else:
                    dataName = 'Unknown'
                for j in range(len(dataType)):
                    metricType = dataType[j]['Metric_Type']
                    values = dataType[j]['Count']
                    
                    #print('Platform:', platformName)
                    Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                    OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                    PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                    ISBN = pd.concat([ISBN, pd.Series([isbnName])], ignore_index = True)
                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Data Type:', dataType)
                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                    AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)    
    
    
    longUsage = pd.concat([Title, OnlineISSN, PrintISSN, ISBN, Publisher, Platform, AccessType, DataType, MetricType, Month, Usage, Vendor], axis = 1)
    
    longUsage.columns = ['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Publisher', 'Platform', 'AccessType', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']


# Multimedia Item Report Downloads - IR_M1
if reportType == 'IRM1' or reportType == 'irm1':      
    if functionVersion == 'Attribute_Performance':
        for h in range(len(dataItems)):
            attributes = dataItems[0]['Items'][h]['Attribute_Performance']
            itemName = dataItems[0]['Items'][h]['Item']
            for i in range(len(attributes)):
                performanceList = attributes[i]['Performance']
                dataType = attributes[i]['Data_Type']
                for j in range(len(performanceList)):
                    dictionaryItemsJ = performanceList.items()
                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                    metricType = keys[j]
                    months = performanceList[keys[j]]
                    for k in range(len(months)):
                        dictionaryItemsK = months.items()
                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                        month = keys2[k]
                        values = months[keys2[k]]
                        
                        #print('Platform:', platformName)
                    Item = pd.concat([Item, pd.Series([itemName])], ignore_index = True)
                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)  
    
    
    if functionVersion == 'Performance':
        for h in range(len(dataItems)):
            performances = dataItems[h]['Performance']
            itemName = dataItems[h]['Item']
            platformName = dataItems[h]['Platform']
            publisherName = dataItems[h]['Publisher']
            for i in range(len(performances)):
                performanceList = performances
                month = performances[i]['Period']['Begin_Date']
                dataType = performances[i]['Instance']
                for j in range(len(dataType)):
                    metricType = dataType[j]['Metric_Type']
                    values = dataType[j]['Count']
                    
                    #print('Platform:', platformName)
                    Item = pd.concat([Item, pd.Series([itemName])], ignore_index = True)
                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                    #print('Metric Type:', metricType)
                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                    #print('Month:', month)
                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                    #print('Usage Value:', values)
                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
    
                    Vendor = pd.concat([Vendor, pd.Series([vendor])], ignore_index = True)    
    
    
    longUsage = pd.concat([Item, Publisher, Platform, MetricType, Month, Usage, Vendor], axis = 1)
    
    longUsage.columns = ['Item', 'Publisher', 'Platform', 'MetricType', 'Month', 'Usage', 'Vendor']

In [44]:
longUsage

,Title,Online_ISSN,Print_ISSN,ISBN,Publisher,Platform,AccessType,DataType,MetricType,Month,Usage,Vendor
0,A Dictionary of,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Total_Item_Investigations,2026-07,1,OUP
1,A Dictionary of,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Total_Item_Requests,2026-07,1,OUP
2,A Dictionary of,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Item_Investigations,2026-07,1,OUP
3,A Dictionary of,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Item_Requests,2026-07,1,OUP
4,A Dictionary of,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Title_Investigations,2026-07,1,OUP
...,...,...,...,...,...,...,...,...,...,...,...,...
2554,World Encyclopedia,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Total_Item_Requests,2026-08,7,OUP
2555,World Encyclopedia,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Item_Investigations,2026-08,6,OUP
2556,World Encyclopedia,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Item_Requests,2026-08,6,OUP
2557,World Encyclopedia,NaN,NaN,NaN,Oxford University Press,Oxford Reference,Controlled,NaN,Unique_Title_Investigations,2026-08,5,OUP


## Convert the Long Data into Wide Format

So far, we have one row per month and item. These next cells create a file looking like it would be downloaded from the vendor website, where each column is a month.

In [45]:
longUsage = longUsage.dropna(axis=1, how='all')

valueColumn = 'Usage'
monthColumn = 'Month'
indexColumns = [col for col in longUsage.columns if col != valueColumn and col != monthColumn]

  
wideUsage = longUsage.pivot_table(index = indexColumns, columns = monthColumn, values = valueColumn).reset_index().fillna(0)

In [46]:
wideUsage

Month,Title,Publisher,Platform,AccessType,MetricType,Vendor,2026-07,2026-08
0,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Total_Item_Investigations,OUP,1.0,0.0
1,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Total_Item_Requests,OUP,1.0,0.0
2,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Investigations,OUP,1.0,0.0
3,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Requests,OUP,1.0,0.0
4,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Title_Investigations,OUP,1.0,0.0
...,...,...,...,...,...,...,...,...
2044,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Total_Item_Requests,OUP,0.0,7.0
2045,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Investigations,OUP,0.0,6.0
2046,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Requests,OUP,0.0,6.0
2047,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Title_Investigations,OUP,0.0,5.0


### Fix Column Names to a Standard Format

Each new semester will need new column naming conventions in both of the dictionaries below.

In [18]:
## Fix Column Names to a Standard Format
wideUsage = wideUsage.rename(columns = {
                            '2021-01':'Jan-2021',
                            '2021-02':'Feb-2021',
                            '2021-03':'Mar-2021',
                            '2021-04':'Apr-2021',
                            '2021-05':'May-2021',
                            '2021-06':'Jun-2021',
                            '2021-07':'Jul-2021',
                            '2021-08':'Aug-2021',
                            '2021-09':'Sep-2021',
                            '2021-10':'Oct-2021',
                            '2021-11':'Nov-2021',
                            '2021-12':'Dec-2021',                            
                            '2022-01':'Jan-2022',
                            '2022-02':'Feb-2022',
                            '2022-03':'Mar-2022',
                            '2022-04':'Apr-2022',
                            '2022-05':'May-2022',
                            '2022-06':'Jun-2022',
                            '2022-07':'Jul-2022',
                            '2022-08':'Aug-2022',
                            '2022-09':'Sep-2022',
                            '2022-10':'Oct-2022',
                            '2022-11':'Nov-2022',
                            '2022-12':'Dec-2022',
                            '2023-01':'Jan-2023',
                            '2023-02':'Feb-2023',
                            '2023-03':'Mar-2023',
                            '2023-04':'Apr-2023',
                            '2023-05':'May-2023',
                            '2023-06':'Jun-2023',
                            '2023-07':'Jul-2023',
                            '2023-08':'Aug-2023',
                            '2023-09':'Sep-2023',
                            '2023-10':'Oct-2023',
                            '2023-11':'Nov-2023',
                            '2023-12':'Dec-2023',                            
                            '2024-01':'Jan-2024',
                            '2024-02':'Feb-2024',
                            '2024-03':'Mar-2024',
                            '2024-04':'Apr-2024',
                            '2024-05':'May-2024',
                            '2024-06':'Jun-2024',
                            '2024-07':'Jul-2024',
                            '2024-08':'Aug-2024',
                            '2024-09':'Sep-2024',
                            '2024-10':'Oct-2024',
                            '2024-11':'Nov-2024',
                            '2024-12':'Dec-2024',
                            '2025-01':'Jan-2025',
                            '2025-02':'Feb-2025',
                            '2025-03':'Mar-2025',
                            '2025-04':'Apr-2025',
                            '2025-05':'May-2025',
                            '2025-06':'Jun-2025',
                            '2025-07':'Jul-2025',
                            '2025-08':'Aug-2025',
                            '2025-09':'Sep-2025',
                            '2025-10':'Oct-2025',
                            '2025-11':'Nov-2025',
                            '2025-12':'Dec-2025',
                            '2026-01':'Jan-2026',
                            '2026-02':'Feb-2026',
                            '2026-03':'Mar-2026',
                            '2026-04':'Apr-2026',
                            '2026-05':'May-2026',
                            '2026-06':'Jun-2026',
                            '2026-07':'Jul-2026',
                            '2026-08':'Aug-2026'})

wideUsage = wideUsage.rename(columns = {
                            '2021-01-01':'Jan-2021',
                            '2021-02-01':'Feb-2021',
                            '2021-03-01':'Mar-2021',
                            '2021-04-01':'Apr-2021',
                            '2021-05-01':'May-2021',
                            '2021-06-01':'Jun-2021',
                            '2021-07-01':'Jul-2021',
                            '2021-08-01':'Aug-2021',
                            '2021-09-01':'Sep-2021',
                            '2021-10-01':'Oct-2021',
                            '2021-11-01':'Nov-2021',
                            '2021-12-01':'Dec-2021',
                            '2022-01-01':'Jan-2022',
                            '2022-02-01':'Feb-2022',
                            '2022-03-01':'Mar-2022',
                            '2022-04-01':'Apr-2022',
                            '2022-05-01':'May-2022',
                            '2022-06-01':'Jun-2022',
                            '2022-07-01':'Jul-2022',
                            '2022-08-01':'Aug-2022',
                            '2022-09-01':'Sep-2022',
                            '2022-10-01':'Oct-2022',
                            '2022-11-01':'Nov-2022',
                            '2022-12-01':'Dec-2022',
                            '2023-01-01':'Jan-2023',
                            '2023-02-01':'Feb-2023',
                            '2023-03-01':'Mar-2023',
                            '2023-04-01':'Apr-2023',
                            '2023-05-01':'May-2023',
                            '2023-06-01':'Jun-2023',
                            '2023-07-01':'Jul-2023',
                            '2023-08-01':'Aug-2023',
                            '2023-09-01':'Sep-2023',
                            '2023-10-01':'Oct-2023',
                            '2023-11-01':'Nov-2023',
                            '2023-12-01':'Dec-2023',
                            '2024-01-01':'Jan-2024',
                            '2024-02-01':'Feb-2024',
                            '2024-03-01':'Mar-2024',
                            '2024-04-01':'Apr-2024',
                            '2024-05-01':'May-2024',
                            '2024-06-01':'Jun-2024',
                            '2024-07-01':'Jul-2024',
                            '2024-08-01':'Aug-2024',
                            '2024-09-01':'Sep-2024',
                            '2024-10-01':'Oct-2024',
                            '2024-11-01':'Nov-2024',
                            '2024-12-01':'Dec-2024',
                            '2025-01-01':'Jan-2025',
                            '2025-02-01':'Feb-2025',
                            '2025-03-01':'Mar-2025',
                            '2025-04-01':'Apr-2025',
                            '2025-05-01':'May-2025',
                            '2025-06-01':'Jun-2025',
                            '2025-07-01':'Jul-2025',
                            '2025-08-01':'Aug-2025',
                            '2025-09-01':'Sep-2025',
                            '2025-10-01':'Oct-2025',
                            '2025-11-01':'Nov-2025',
                            '2025-12-01':'Dec-2025',
                            '2026-01-01':'Jan-2026',
                            '2026-02-01':'Feb-2026',
                            '2026-03-01':'Mar-2026',
                            '2026-04-01':'Apr-2026',
                            '2026-05-01':'May-2026',
                            '2026-06-01':'Jun-2026',
                            '2026-07-01':'Jul-2026',
                            '2026-08-01':'Aug-2026'})

# Add the Reporting_Period_Total to the final table
if reportType == 'DR' or reportType == 'dr':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
elif reportType == 'PR' or reportType == 'pr':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 4:].sum(axis = 1)
elif reportType == 'TR' or reportType == 'tr':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
elif reportType == 'TRB3' or reportType == 'trb3':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
elif reportType == 'TRJ3' or reportType == 'trj3':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
elif reportType == 'IRM1' or reportType == 'irm1':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
elif reportType == 'DRD1' or reportType == 'drd1':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
elif reportType == 'PRP1' or reportType == 'prp1':
    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 4:].sum(axis = 1)

wideUsage

Month,Title,Publisher,Platform,AccessType,MetricType,Vendor,Jul-2026,Aug-2026,Reporting_Period_Total
0,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Total_Item_Investigations,OUP,1.0,0.0,0.0
1,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Total_Item_Requests,OUP,1.0,0.0,0.0
2,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Investigations,OUP,1.0,0.0,0.0
3,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Requests,OUP,1.0,0.0,0.0
4,A Dictionary of,Oxford University Press,Oxford Reference,Controlled,Unique_Title_Investigations,OUP,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2044,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Total_Item_Requests,OUP,0.0,7.0,0.0
2045,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Investigations,OUP,0.0,6.0,0.0
2046,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Item_Requests,OUP,0.0,6.0,0.0
2047,World Encyclopedia,Oxford University Press,Oxford Reference,Controlled,Unique_Title_Investigations,OUP,0.0,5.0,0.0


### If the wideUsage has much fewer or no rows compared to the longUsage, then there is likely a column with NA values in the index. This column(s) should be removed and the rows will re-appear.

## Save the Usage Files to the Usage Folder


#### Create a standard file name based on the date range

If you are using a start or end month not already in the dictionaries below, make sure to add it before continuing.

In [19]:
## Create the Date Piece for Export Name Based on the Initial Date

renameStartDateDict = {'2021-01-01':'Jan2021',
                            '2021-03-01':'Mar2021',
                            '2021-05-01':'May2021',
                            '2021-07-01':'Jul2021',
                            '2021-08-01':'Aug2021',
                            '2022-01-01':'Jan2022',
                            '2022-02-01':'Feb2022',
                            '2022-07-01':'Jul2022',
                            '2023-01-01':'Jan2023',
                            '2023-02-01':'Feb2023',
                            '2023-03-01':'Mar2023',
                            '2023-04-01':'Apr2023',
                            '2023-05-01':'May2023',
                            '2023-06-01':'Jun2023',
                            '2023-07-01':'Jul2023',
                            '2023-08-01':'Aug2023',
                            '2023-09-01':'Sep2023',
                            '2023-10-01':'Oct2023',
                            '2023-11-01':'Nov2023',
                            '2023-12-01':'Dec2023',
                            '2024-01-01':'Jan2024',
                            '2024-02-01':'Feb2024',
                            '2024-03-01':'Mar2024',
                            '2024-04-01':'Apr2024',
                            '2024-05-01':'May2024',
                            '2024-06-01':'Jun2024',
                            '2024-07-01':'Jul2024',
                            '2024-08-01':'Aug2024',
                            '2024-09-01':'Sep2024',
                            '2024-10-01':'Oct2024',
                            '2024-11-01':'Nov2024',
                            '2024-12-01':'Dec2024',
                            '2025-01-01':'Jan2025',
                            '2025-02-01':'Feb2025',
                            '2025-03-01':'Mar2025',
                            '2025-04-01':'Apr2025',
                            '2025-05-01':'May2025',
                            '2025-06-01':'Jun2025',
                            '2025-07-01':'Jul2025',
                            '2025-08-01':'Aug2025',
                            '2025-09-01':'Sep2025',
                            '2025-10-01':'Oct2025',
                            '2025-11-01':'Nov2025',
                            '2025-12-01':'Dec2025',
                            '2026-01-01':'Jan2026',
                            '2026-02-01':'Feb2026',
                            '2026-07-01':'Jul2026'}

renameEndDateDict = {'2021-06-30':'Jun2021',
                            '2021-12-31':'Dec2021',
                            '2022-06-30':'Jun2022',
                            '2022-12-31':'Dec2022',
                            '2023-01-31':'Jan2023',
                            '2023-02-28':'Feb2023',
                            '2023-03-31':'Mar2023',
                            '2023-04-30':'Apr2023',
                            '2023-05-31':'May2023',
                            '2023-06-30':'Jun2023',
                            '2023-10-31':'Oct2023',
                            '2023-11-30':'Nov2023',
                            '2023-12-31':'Dec2023',
                            '2024-01-31':'Jan2024',
                            '2024-02-29':'Feb2024',
                            '2024-03-31':'Mar2024',
                            '2024-04-30':'Apr2024',
                            '2024-05-31':'May2024',
                            '2024-06-30':'Jun2024',
                            '2024-07-31':'Jul2024',
                            '2024-08-31':'Aug2024',
                            '2024-09-30':'Sep2024',
                            '2024-10-31':'Oct2024',
                            '2024-11-30':'Nov2024',
                            '2024-12-31':'Dec2024',
                            '2025-01-31':'Jan2025',
                            '2025-02-28':'Feb2025',
                            '2025-03-31':'Mar2025',
                            '2025-04-30':'Apr2025',
                            '2025-05-31':'May2025',
                            '2025-06-30':'Jun2025',
                            '2025-10-31':'Oct2025',
                            '2025-11-30':'Nov2025',
                            '2025-12-31':'Dec2025',
                            '2026-01-31':'Jan2026',
                            '2026-03-31':'Mar2026',
                            '2026-05-31':'May2026',
                            '2026-06-30':'Jun2026',
                            '2026-08-31':'Aug2026'}

renameStartDateDict[startDate]

'Jul2026'

### Export the Usage

The usage will appear in the Python folder this code is saved to by default. If you'd rather save it locally, you can set up the location in the code box below.

In [20]:
folderPath = r"C:\Users\aharrima\GitHub_UTK\Usage_Database_Holding_Area" # Change this path for your computer as needed. If you don't need to save locally, comment this out.

directory = os.chdir(folderPath)

In [21]:
wideUsage.to_excel(r''+vendor+'-'+reportType+'-'+renameStartDateDict[startDate]+'-'+renameEndDateDict[endDate]+'.xlsx', index = False)

## Next Steps:

### Gathering Usage in Bulk?

Then move on to the next report. If you feel that all of the reports should download properly, you can also use the "Gathering Lots of SUSHI Straight from COUNTER API" file, which allows for multiple report gathering.


### Updating the COUNTER 5/5.1 Database?

Take this usage and apply them to the "Uploading COUNTER Data to SQLite3" file. Make sure all of these files are in a folder named "From Python" to ensure proper loading into "Uploading COUNTER Data."


### Ready to upload usage?

Place it in its proper usage folder(s) in SharePoint. If this is part of a semi-annual bulk usage gathering, make sure to mark off these vendors as gathered in Usage Master Vendor Report.